# Function 7: 6D black-box maximisation

You’re tasked with optimising an ML model by tuning six hyperparameters, for example learning rate, regularisation strength or number of hidden layers. The function you’re maximising is the model’s performance score (such as accuracy or F1), but since the relationship between inputs and output isn’t known, it’s treated as a black-box function. Because this is a commonly used model, you might benefit from researching best practices or literature to guide your initial search space. Your goal is to find the combination of hyperparameters that yields the highest possible performance.

## How this notebook works

1. **Data loading:** observations are read from `data.csv`; the incumbent is the best observed `y`.
2. **Gaussian process:** an exact GPyTorch GP (constant mean, scaled Matérn 5/2 ARD kernel, bounded noise), fitted with Adam on standardised `y`; it uses Apple MPS or CUDA when available.
3. **Trust-region replay:** the TuRBO box length is rebuilt from the weekly history: it doubles after `SUCC_TOL` consecutive improvements and halves after `FAIL_TOL` failures.
4. **Candidates:** Sobol points in an ARD-shaped box centred on the observed point with the highest fitted GP mean, plus perturbations of the incumbent.
5. **Scoring:** the posterior-mean maximiser (exploitation) and a marginal Thompson-style draw (exploration).
6. **Candidates:** both are printed in submission format with their GP predictions.
7. **Final submission:** the GP predicts a decline from the best point (R11), so a quadratic is fitted through rounds 10, 11 and 12 (checked to be equally spaced along one line) and its vertex, truncated to 6 decimals, is the final query.

Everything is derived from `data.csv`: append new rows and re-run, with no code edits. Change `DATA_PATH` only if the notebook is in a different directory.

**Data-handling choices.**

- *Thompson-style draw.* It samples each candidate independently and is not a correlated Thompson sample.
- *Trust region.* TuRBO is reconstructed from the history, not maintained by a fully sequential run.
- *Input format.* Candidates are truncated (not rounded) to 6 decimals, as the portal requires. Nothing is submitted automatically.

In [1]:
# =============================================================================
# IMPORTS AND SETTINGS
# =============================================================================

import warnings
from decimal import Decimal, ROUND_DOWN

import numpy as np
import pandas as pd
from scipy.stats import qmc
import torch
import gpytorch

warnings.filterwarnings("ignore", category=UserWarning)

DATA_PATH = "../data/data.csv"
FUNCTION_ID = 7
SEED = 0

# TuRBO trust region
LENGTH_INIT = 0.20     # initial box side length (fraction of the unit cube)
LENGTH_MIN = 0.5 ** 7  # box is reset to LENGTH_INIT if it shrinks below this
LENGTH_MAX = 0.50
SUCC_TOL = 3           # consecutive improving rounds before the box doubles
FAIL_TOL = None        # None -> max(4, d) failing rounds before the box halves
SUCCESS_EPS = 1e-6     # minimum gain that counts as an improvement

# Candidate generation
N_CAND = 16384         # Sobol candidates inside the trust region
N_LOCAL = 4096         # Gaussian perturbations around the incumbent
LOCAL_SCALE = 0.025    # std of those perturbations

# GP fitting (Adam on the exact marginal log-likelihood)
N_GP_STEPS = 500
GP_LR = 0.05
DTYPE = torch.float32

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available()
          else torch.device("cpu"))
print(f"PyTorch device: {DEVICE}")

PyTorch device: mps


In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

# round_trip parsing keeps float64 values exactly as written in the CSV
data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [
    f"x{i}" for i in range(1, 9)
    if function_data[f"x{i}"].notna().any()
]

# Modelling arrays
X = function_data[feature_cols].to_numpy()
y = function_data["y"].to_numpy()
rounds = function_data["round"].to_numpy()

# Display dataframe
df = function_data[[*feature_cols, "y", "round"]].copy()
df["Stage"] = df["round"].apply(
    lambda r: "Initial Data" if r == 0 else f"Week {r}"
)
df = df.rename(columns={"round": "Round"})
df = df[[*feature_cols, "y", "Stage", "Round"]]

# Display formatting only — underlying float64 values are unchanged
pd.set_option(
    "display.float_format",
    lambda v: f"{v:.6e}" if abs(v) < 1e-3 and v != 0 else f"{v:.6f}"
)

print(f"Function {FUNCTION_ID} observations: {len(df)}")
print(f"Active features: {feature_cols}")
print(f"Dimensions: {len(feature_cols)} | X: {X.shape} | y: {y.shape}")
print(f"Rounds: R0–R{df['Round'].max()}")
display(df)

Function 7 observations: 42
Active features: ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
Dimensions: 6 | X: (42, 6) | y: (42,)
Rounds: R0–R12


,x1,x2,x3,x4,x5,x6,y,Stage,Round
105,0.272624,0.324495,0.897109,0.832951,0.154063,0.795864,0.604433,Initial Data,0
106,0.543003,0.924694,0.341567,0.646486,0.718440,0.343133,0.562753,Initial Data,0
107,0.090832,0.661529,0.065931,0.258577,0.963453,0.640265,0.007503,Initial Data,0
108,0.118867,0.615055,0.905816,0.855300,0.413631,0.585236,0.061424,Initial Data,0
109,0.630218,0.838097,0.680013,0.731895,0.526737,0.348429,0.273047,Initial Data,0
110,0.764919,0.255883,0.609084,0.218079,0.322943,0.095794,0.083747,Initial Data,0
111,0.057896,0.491672,0.247422,0.218118,0.420428,0.730970,1.364968,Initial Data,0
112,0.195252,0.079227,0.554580,0.170567,0.014944,0.107032,0.092645,Initial Data,0
113,0.642303,0.836875,0.021793,0.101488,0.683071,0.692416,0.017870,Initial Data,0
114,0.789943,0.195545,0.575623,0.073659,0.259049,0.051100,0.033565,Initial Data,0


In [3]:
# =============================================================================
# HELPERS
# =============================================================================

def truncate6(v):
    """Truncate (not round) to six decimals, as required by the submission format."""
    return str(Decimal(repr(float(v))).quantize(Decimal("0.000001"), rounding=ROUND_DOWN))


class ExactARDGP(gpytorch.models.ExactGP):
    """Exact GP: constant mean, scaled Matérn-5/2 kernel with one length scale per input."""

    def __init__(self, train_x, train_y, likelihood, d):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=2.5, ard_num_dims=d,
                lengthscale_constraint=gpytorch.constraints.Interval(0.01, 10.0)),
            outputscale_constraint=gpytorch.constraints.Interval(1e-2, 1e2))

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x))

In [4]:
# =============================================================================
# TuRBO-1 CANDIDATE GENERATION
# =============================================================================
# 1. Fit the ARD Matérn GP to all observations (standardised y).
# 2. Replay the weekly history to rebuild the trust-region box length
#    (double after SUCC_TOL successes, halve after FAIL_TOL failures).
# 3. Centre an ARD-shaped box on the observed point with the highest GP mean.
# 4. Generate Sobol candidates inside the box plus perturbations of the incumbent.
# 5. Pick the posterior-mean maximiser and a marginal Thompson-style draw.

def turbo_next(X, y, rounds, x_cols, seed=0):
    n, d = X.shape
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    fail_tol = max(4, d) if FAIL_TOL is None else FAIL_TOL

    # 1. Fit the GP on all observations, including the initial design
    y_mu, y_sd = y.mean(), y.std()
    train_x = torch.tensor(X, dtype=DTYPE, device=DEVICE)
    train_y = torch.tensor((y - y_mu) / y_sd, dtype=DTYPE, device=DEVICE)
    likelihood = gpytorch.likelihoods.GaussianLikelihood(
        noise_constraint=gpytorch.constraints.Interval(1e-6, 1.0)).to(DEVICE)
    model = ExactARDGP(train_x, train_y, likelihood, d).to(DEVICE)
    model.covar_module.base_kernel.lengthscale = 0.3
    model.covar_module.outputscale = 1.0
    likelihood.noise = 0.1

    model.train(); likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=GP_LR)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    for _ in range(N_GP_STEPS):
        optimizer.zero_grad()
        loss = -mll(model(train_x), train_y)
        loss.backward()
        optimizer.step()
    model.eval(); likelihood.eval()

    ls = model.covar_module.base_kernel.lengthscale.detach().cpu().numpy().reshape(-1)
    noise_sd = np.sqrt(float(likelihood.noise.item())) * y_sd

    # 2. Replay the weekly history to reconstruct the trust-region length
    order = np.argsort(rounds, kind="stable")
    r_s, y_s = rounds[order], y[order]
    first = r_s.min()
    length, succ, fail = LENGTH_INIT, 0, 0
    best = y_s[r_s == first].max()

    print("Round  best y in round   outcome   box length after")
    for r in np.unique(r_s[r_s > first]):
        yr = y_s[r_s == r].max()
        if yr > best + SUCCESS_EPS:
            succ += 1; fail = 0; best = yr; outcome = "success"
        else:
            succ = 0; fail += 1; outcome = "failure"
        if succ >= SUCC_TOL:
            length = min(2.0 * length, LENGTH_MAX); succ = 0
        elif fail >= fail_tol:
            length /= 2.0; fail = 0
        if length < LENGTH_MIN:
            length = LENGTH_INIT; succ = 0; fail = 0
        print(f"{r:>5}  {yr:>15.9f}   {outcome:<8}  {length:.4f}")

    # 3. ARD-shaped trust region centred on the observed point with highest GP mean
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        obs_mean = likelihood(model(train_x)).mean.detach().cpu().numpy()
    centre = X[np.argmax(obs_mean)].copy()
    incumbent = X[np.argmax(y)].copy()

    w = ls / ls.mean()
    w = w / np.prod(w) ** (1.0 / d)
    lb = np.clip(centre - w * length / 2, 0, 1)
    ub = np.clip(centre + w * length / 2, 0, 1)

    # 4. Candidates: Sobol points (each perturbing a random subset of
    #    coordinates), perturbations of the incumbent, and the incumbent itself
    sobol = qmc.Sobol(d=d, scramble=True, seed=seed)
    pert = lb + (ub - lb) * sobol.random(N_CAND)
    mask = rng.random((N_CAND, d)) <= min(1.0, 20.0 / d)
    empty = np.where(mask.sum(axis=1) == 0)[0]
    if len(empty):
        mask[empty, rng.integers(0, d, size=len(empty))] = True
    cand_tr = np.where(mask, pert, centre)

    local = incumbent + rng.normal(0, LOCAL_SCALE, size=(N_LOCAL, d))
    local = np.clip(np.clip(local, lb, ub), 0, 1)
    candidates = np.vstack([cand_tr, local, incumbent[None, :]])

    # 5. Posterior predictions (including observation noise)
    candidate_x = torch.tensor(candidates, dtype=DTYPE, device=DEVICE)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        candidate_pred = likelihood(model(candidate_x))
        mu_std = candidate_pred.mean.detach().cpu().numpy()
    mu_y = mu_std * y_sd + y_mu

    # Marginal independent Thompson-style draw (NOT a correlated posterior sample)
    eps = torch.randn(len(candidates), dtype=DTYPE, device=DEVICE)
    with torch.no_grad():
        draw = candidate_pred.mean + candidate_pred.variance.sqrt() * eps
        ts_idx = int(torch.argmax(draw).item())
    mean_idx = int(np.argmax(mu_y))

    # Truncate both candidates to submission format and re-evaluate them
    def truncated(idx):
        strings = [truncate6(v) for v in candidates[idx]]
        return strings, np.array([float(s) for s in strings])

    def predict_one(x):
        xt = torch.tensor(x[None, :], dtype=DTYPE, device=DEVICE)
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred = likelihood(model(xt))
            return (float(pred.mean.item()) * y_sd + y_mu,
                    float(pred.variance.sqrt().item()) * y_sd)

    ts_str, x_ts = truncated(ts_idx)
    mean_str, x_mean = truncated(mean_idx)
    ts_mean, ts_sd = predict_one(x_ts)
    mean_pred, mean_sd = predict_one(x_mean)

    # 6. Diagnostics
    print(f"\nObservations: {n} | Inputs: {x_cols}")
    print(f"Best observed y: {y.max():.9f} at {np.round(incumbent, 6)}")
    print(f"GP fitted noise SD (y units): {noise_sd:.8f}")
    print("Length scales: " + ", ".join(f"{c}={v:.3f}" for c, v in zip(x_cols, ls)))
    print(f"Box length: {length:.4f} (succ {succ}/{SUCC_TOL}, fail {fail}/{fail_tol})")
    print(f"Centre (best observed GP mean): {np.round(centre, 6)}")
    print("\nTrust region:")
    for c, lo, hi in zip(x_cols, lb, ub):
        print(f"  {c}: [{lo:.4f}, {hi:.4f}]")

    print("\nBest posterior-mean candidate (exploitation):")
    print("-".join(mean_str))
    print(f"Predicted y: {mean_pred:.9f} +/- {mean_sd:.9f}")
    print(f"Predicted gain: {mean_pred - y.max():+.9f}")
    print(f"Distance from incumbent: {np.linalg.norm(x_mean - incumbent):.6f}")

    print("\nThompson-style candidate (exploration, independent marginal draw):")
    print("-".join(ts_str))
    print(f"Predicted y: {ts_mean:.9f} +/- {ts_sd:.9f}")
    print(f"Distance from incumbent: {np.linalg.norm(x_ts - incumbent):.6f}")

    return {"x_next": x_ts, "x_next_str": ts_str,
            "x_mean": x_mean, "x_mean_str": mean_str,
            "centre": centre, "incumbent": incumbent,
            "length": length, "lb": lb, "ub": ub,
            "ts_predicted_y": ts_mean, "mean_predicted_y": mean_pred}

In [5]:
# =============================================================================
# RUN
# =============================================================================

result4 = turbo_next(X, y, rounds, feature_cols, seed=SEED)
x_next4 = result4["x_next"]              # Marginal Thompson-style candidate
x_next4_str = result4["x_next_str"]
x_mean4 = result4["x_mean"]              # Posterior-mean exploitation candidate
x_mean4_str = result4["x_mean_str"]

Round  best y in round   outcome   box length after
    1      2.308609380   success   0.2000
    2      1.227023912   failure   0.2000
    3      1.244834084   failure   0.2000
    4      2.384264926   success   0.2000
    5      2.392538968   success   0.2000
    6      2.421721948   success   0.4000
    7      2.828503131   success   0.4000
    8      2.853373462   success   0.4000
    9      3.240233640   success   0.5000
   10      3.267412601   success   0.5000
   11      3.283106283   success   0.5000
   12      3.272948457   failure   0.5000

Observations: 42 | Inputs: ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
Best observed y: 3.283106283 at [0.229484 0.104363 0.480638 0.263197 0.314339 0.651216]
GP fitted noise SD (y units): 0.00509658
Length scales: x1=0.548, x2=0.599, x3=8.055, x4=0.402, x5=0.214, x6=0.325
Box length: 0.5000 (succ 0/3, fail 1/6)
Centre (best observed GP mean): [0.229484 0.104363 0.480638 0.263197 0.314339 0.651216]

Trust region:
  x1: [0.0179, 0.4411]
  x2: [0.0

In [7]:
# =============================================================================
# FUNCTION 7 — FINAL SUBMISSION: QUADRATIC INTERPOLATION
# =============================================================================
# TuRBO predicted a decline from the best result in Round 11, suggesting
# that an optimum is being approached. Since performance improved through
# Round 11 before declining in Round 12. Quadratic interpolation was used
# to estimate a potential peak between the two rounds.
# Fit a quadratic through Rounds 10, 11 and 12.
# =============================================================================

# 1. Extract actual observations
rounds = [10, 11, 12]

recent = (
    function_data[
        function_data["round"].isin(rounds)
    ]
    .sort_values("round")
    .copy()
)

if recent["round"].tolist() != rounds:
    raise ValueError("Expected exactly one observation for Rounds 10–12.")

X_recent = recent[feature_cols].to_numpy(dtype=float)
y_recent = recent["y"].to_numpy(dtype=float)

# 2. Verify that the three points are equally spaced
step_1 = X_recent[1] - X_recent[0]
step_2 = X_recent[2] - X_recent[1]

if not np.allclose(step_1, step_2, atol=1e-6):
    raise ValueError("Rounds 10–12 are not equally spaced.")

# 3. Fit quadratic: y(t) = a*t² + b*t + c
#    Round 10: t = -1
#    Round 11: t =  0
#    Round 12: t = +1

t = np.array([-1.0, 0.0, 1.0])

a, b, c = np.polyfit(t, y_recent, 2)

if a >= 0:
    raise ValueError("Quadratic does not have a maximum.")

# 4. Calculate vertex (estimated maximum)
t_star = -b / (2 * a)

x_star = X_recent[1] + t_star * step_2
y_star = np.polyval([a, b, c], t_star)

# 5. Format submission to six decimal places
x_submit = np.floor(x_star * 1e6) / 1e6

submission = "-".join(
    f"{v:.6f}" for v in x_submit
)

# 6. Results
print("FUNCTION 7 — QUADRATIC INTERPOLATION")
print("=" * 55)

print("\nObserved results:")
for r, y_value in zip(rounds, y_recent):
    print(f"  Round {r}: {y_value:.9f}")

print(f"\nQuadratic curvature (a): {a:.9f}")
print(f"Estimated vertex (t):    {t_star:.6f}")

print(f"\nCurrent best y:          {y_recent[1]:.9f}")
print(f"Interpolated maximum:    {y_star:.9f}")
print(f"Estimated improvement:   {y_star - y_recent[1]:+.9f}")

print("\nProposed final submission:")
print(submission)

FUNCTION 7 — QUADRATIC INTERPOLATION

Observed results:
  Round 10: 3.267412601
  Round 11: 3.283106283
  Round 12: 3.272948457

Quadratic curvature (a): -0.012925754
Estimated vertex (t):    0.107070

Current best y:          3.283106283
Interpolated maximum:    3.283254464
Estimated improvement:   +0.000148181

Proposed final submission:
0.230117-0.105291-0.483789-0.263529-0.313756-0.650578
